# Registro histórico: 10_partial_finetuning

Origem: `C52` — `Colab Notebooks/Untitled24.ipynb`.

EfficientNet com descongelamento parcial e até 150 épocas.

**Limitações conhecidas:** Treino interrompido, mas há saída de teste; pos_weight e mapeamento precisam de revisão.

Cópia organizada em 25/10/2026. Código e texto originais preservados abaixo; saídas, contagens de execução e metadados do Colab removidos. As métricas salvas estão em `results/historical_evidence.json`. Não executar em lote: estes cadernos dependem do ambiente e de arquivos originais. Veja `docs/experiments.md` para o fluxo novo.


In [ ]:
import os
import random
import shutil
from pathlib import Path
from timeit import default_timer as timer

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import requests

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.models as models
from torchvision import transforms, datasets
from torchvision.models import EfficientNet_B0_Weights

In [ ]:
# ==============================================================================
# 1. Configuração e Download de Dados (Mantido)
# ==============================================================================

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Set random seeds for reproducibility
SEED = 91432
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Diretório onde o Kaggle vai salvar o dataset
DATASET_DIR = "data_ham10000"

os.makedirs(DATASET_DIR, exist_ok=True)

# Verifica se o arquivo de credenciais existe
kaggle_config_path = "kaggle.json"
print("kaggle.json existe?", os.path.exists(kaggle_config_path))

# Simulação de comandos para download do dataset (necessário para rodar o código)
# Estes comandos assumem que o arquivo kaggle.json está presente no ambiente.
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p {DATASET_DIR} --force

# Descompacta o arquivo
import zipfile
from glob import glob

zip_files = glob(os.path.join(DATASET_DIR, "*.zip"))
for zf in zip_files:
    print("Descompactando:", zf)
    with zipfile.ZipFile(zf, 'r') as zip_ref:
        zip_ref.extractall(DATASET_DIR)

import pandas as pd
import os

# Caminho do arquivo de metadados
metadata_path = os.path.join(DATASET_DIR, "HAM10000_metadata.csv")

# Lê o CSV de metadados
try:
    df = pd.read_csv(metadata_path)
except FileNotFoundError:
    print(f"ERRO: Arquivo de metadados não encontrado em {metadata_path}. Certifique-se de que o dataset foi baixado e descompactado corretamente.")
    # Cria um DataFrame de exemplo para evitar erro de código, mas o treinamento falhará
    df = pd.DataFrame({
        'image_id': ['ISIC_0000000', 'ISIC_0000001'],
        'dx': ['nv', 'mel']
    })


# Pastas onde estão as imagens
img_dir_1 = os.path.join(DATASET_DIR, "ham10000_images_part_1")
img_dir_2 = os.path.join(DATASET_DIR, "ham10000_images_part_2")

In [ ]:
# ==============================================================================
# 2. Pré-processamento e Divisão dos Dados (Mantido)
# ==============================================================================

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset

# cópia do df original
df_binary = df.copy()

# coluna binária: 1 para 'mel', 0 para 'nonMel'
df_binary["target"] = (df_binary["dx"] == "mel").astype(int)

def get_image_path(image_id):
    filename = image_id + ".jpg"
    path1 = os.path.join(img_dir_1, filename)
    path2 = os.path.join(img_dir_2, filename)
    if os.path.exists(path1):
        return path1
    elif os.path.exists(path2):
        return path2
    else:
        return None

df_binary["filepath"] = df_binary["image_id"].apply(get_image_path)

# remove possíveis linhas sem arquivo
df_binary = df_binary.dropna(subset=["filepath"]).reset_index(drop=True)

# Primeiro split: separa treino (70%) e temporary (30%)
train_df, temp_df = train_test_split(
    df_binary,
    test_size=0.3,
    random_state=SEED,
    stratify=df_binary["target"]
)

# Segundo split: separa validação (15%) e teste (15%) do temporary
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,  # 50% de 30% = 15% do total
    random_state=SEED,
    stratify=temp_df["target"]
)

In [ ]:
# ==============================================================================
# 3. Cálculo do Peso da Classe (Aprimorado)
# ==============================================================================

# Calcular o peso da classe para a função de perda (BCEWithLogitsLoss)
# O peso deve ser um tensor de float para a classe positiva (melanoma)
# O peso é calculado com base no desbalanceamento do conjunto de TREINO.
y_train = train_df["target"].values
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# O pos_weight é o peso da classe positiva (melanoma, índice 1)
pos_weight_value = class_weights[1]
pos_weight = torch.tensor(pos_weight_value, dtype=torch.float).to(device)

print(f"Peso da classe positiva (melanoma) calculado: {pos_weight_value:.4f}")

In [ ]:
# ==============================================================================
# 4. Definição do Dataset e Transformações (Aprimorado: Adição de TTA)
# ==============================================================================

class SkinLesionDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["filepath"]
        label = row["target"]

        # Carregar imagem e garantir que está em RGB
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float)

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 8

# Pesos pré-treinados (ImageNet)
weights = EfficientNet_B0_Weights.IMAGENET1K_V1
transform = weights.transforms()
mean = transform.mean
std = transform.std

# Transformações de Treino (Data Augmentation Pesada)
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Transformações de Validação/Teste (Apenas Resize e Normalização)
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Dataset e DataLoader
train_dataset = SkinLesionDataset(train_df, transform=train_transform)
val_dataset = SkinLesionDataset(val_df, transform=val_test_transform)
test_dataset = SkinLesionDataset(test_df, transform=val_test_transform)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

In [ ]:
# ==============================================================================
# 5. Definição do Modelo (Mantido)
# ==============================================================================

# Carregar EfficientNet-B0 pré-treinado
base_model = models.efficientnet_b0(weights=weights)

# Congelar a maioria das camadas
for param in base_model.parameters():
    param.requires_grad = False

# Descongelar as últimas 5 blocos de camadas (features) para fine-tuning
# O código original usava base_model.features[:-5], o que congela as últimas 5.
# Para descongelar as últimas 5, deve-se usar base_model.features[-5:]
# No entanto, o código original parecia querer congelar *tudo exceto* o classificador.
# Vamos manter a lógica original de congelar tudo, exceto o classificador,
# e apenas descongelar as últimas camadas do *classificador* para fine-tuning.

# Substituir o classificador para a tarefa binária (2 classes)
num_ftrs = base_model.classifier[1].in_features
base_model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features=num_ftrs, out_features=1) # Saída única para classificação binária
)

# Descongelar as últimas camadas do classificador para fine-tuning
for param in base_model.classifier.parameters():
    param.requires_grad = True

# Descongelar as últimas 5 camadas do bloco de features para fine-tuning
# Esta é uma estratégia comum de fine-tuning.
for param in base_model.features[-5:].parameters():
    param.requires_grad = True

base_model = base_model.to(device)

In [ ]:
# ==============================================================================
# 6. Otimizador, Função de Perda e Scheduler (Mantido/Aprimorado)
# ==============================================================================

# A função de perda agora usa o pos_weight calculado
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Usar um Learning Rate menor para fine-tuning
optimizer = optim.AdamW(base_model.parameters(),
                       lr=1e-4,
                       weight_decay=1e-4)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer,
                                               mode='min',
                                               factor=0.5,
                                               patience=3)

In [ ]:
# ==============================================================================
# 7. Funções de Treinamento e Validação (Mantido)
# ==============================================================================

# Download helper functions from Learn PyTorch repo (necessário para o código original)
if Path("helper_functions.py").is_file():
    print("helper_functions.py already exists, skipping download")
else:
    print("Downloading helper_functions.py")
    request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py")
    with open("helper_functions.py", "wb") as f:
        f.write(request.content)

from helper_functions import accuracy_fn # Acurácia não é a melhor métrica, mas mantida para consistência

def train_step(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device = device):
    """Enhanced training step with correct binary classification."""
    model.train()
    train_loss, train_acc = 0, 0

    progress_bar = tqdm(data_loader, desc="Training", leave=False)

    for batch, (X, y) in enumerate(progress_bar):
        X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)

        # Forward pass
        y_pred_logits = model(X).squeeze(1)
        loss = loss_fn(y_pred_logits, y)

        # Calcular predições binárias corretamente
        y_pred_probs = torch.sigmoid(y_pred_logits)
        y_pred_class = (y_pred_probs >= 0.5).float()

        # Calcular acurácia
        correct = (y_pred_class == y).sum().item()
        acc = (correct / len(y)) * 100

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Update metrics
        train_loss += loss.item()
        train_acc += acc

        progress_bar.set_postfix({
            'batch_loss': f'{loss.item():.4f}',
            'batch_acc': f'{acc:.2f}%'
        })

    # Calculate average metrics
    train_loss /= len(data_loader)
    train_acc /= len(data_loader)

    return train_loss, train_acc

In [ ]:
def val_step(model: torch.nn.Module,
             data_loader: torch.utils.data.DataLoader,
             loss_fn: torch.nn.Module,
             device: torch.device = device):
    """Enhanced validation step with correct binary classification."""
    model.eval()
    val_loss, val_acc = 0, 0

    progress_bar = tqdm(data_loader, desc="Validation", leave=False)

    with torch.inference_mode():
        for X, y in progress_bar:
            X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)

            # Forward pass
            y_pred_logits = model(X).squeeze(1)
            loss = loss_fn(y_pred_logits, y)

            # Calcular predições binárias corretamente
            y_pred_probs = torch.sigmoid(y_pred_logits)
            y_pred_class = (y_pred_probs >= 0.5).float()

            # Calcular acurácia
            correct = (y_pred_class == y).sum().item()
            acc = (correct / len(y)) * 100

            # Update metrics
            val_loss += loss.item()
            val_acc += acc

            progress_bar.set_postfix({
                'batch_loss': f'{loss.item():.4f}',
                'batch_acc': f'{acc:.2f}%'
            })

    # Calculate average metrics
    val_loss /= len(data_loader)
    val_acc /= len(data_loader)

    return val_loss, val_acc

def print_train_time(start: float, end: float, device: torch.device = None):
    """Print training time."""
    total_time = end - start
    print(f"Train time on {device}: {total_time:.3f} seconds")
    return total_time

In [ ]:
# ==============================================================================
# 8. Loop de Treinamento com Early Stopping (Mantido)
# ==============================================================================

EPOCHS = 150
PATIENCE = 30
MIN_DELTA = 0.01

best_val_acc = 0.0
best_model_state = None
epochs_no_improve = 0

train_losses, train_accs = [], []
val_losses, val_accs = [], []

print("Starting training with Early Stopping...")
print(f"Max epochs: {EPOCHS} | Patience: {PATIENCE} | Min delta: {MIN_DELTA}")
print("-" * 50)

train_start_time = timer()

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 30)

    # Training phase
    train_loss, train_acc = train_step(
        model=base_model,
        data_loader=train_dataloader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        device=device
    )

    # Validation phase
    val_loss, val_acc = val_step(
        model=base_model,
        data_loader=val_dataloader,
        loss_fn=loss_fn,
        device=device
    )

    # Learning rate scheduling
    scheduler.step(val_loss)

    # Store metrics
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    # Print epoch results
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    print(f"Current LR: {optimizer.param_groups[0]['lr']:.6f}")

    # Early Stopping Logic
    if val_acc > best_val_acc + MIN_DELTA:
        best_val_acc = val_acc
        best_model_state = base_model.state_dict().copy()
        epochs_no_improve = 0
        print(f"🎉 New best validation accuracy: {best_val_acc:.2f}%")
    else:
        epochs_no_improve += 1
        print(f"⏳ No improvement for {epochs_no_improve}/{PATIENCE} epochs")

        if epochs_no_improve >= PATIENCE:
            print(f"🛑 Early stopping triggered after {epoch+1} epochs")
            print(f"📊 Best validation accuracy: {best_val_acc:.2f}%")
            break

train_end_time = timer()
total_train_time = print_train_time(train_start_time, train_end_time, device)

print(f"\nTraining completed!")
print(f"Best validation accuracy: {best_val_acc:.2f}%")

In [ ]:
# ==============================================================================
# 9. Avaliação e Relatório Final (Aprimorado: Adição de TTA na avaliação)
# ==============================================================================

def eval_model(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               device: torch.device = device,
               tta_steps: int = 0): # Novo parâmetro para TTA
    """Comprehensive model evaluation with correct binary classification."""
    model.eval()
    test_loss, test_acc = 0, 0
    all_preds, all_labels = [], []

    # TTA transforms (horizontal flip and rotation)
    tta_transforms = [
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.1, contrast=0.1)
    ]

    with torch.inference_mode():
        progress_bar = tqdm(data_loader, desc="Evaluating")
        for X, y in progress_bar:
            X, y = X.to(device), y.to(device)

            # 1. Forward pass normal
            y_pred_logits_sum = model(X).squeeze(1)

            # 2. Test Time Augmentation (TTA)
            if tta_steps > 0:
                for _ in range(tta_steps):
                    # Aplicar transformações TTA (simplesmente re-aplicando as transformações de treino)
                    # Para TTA, o ideal é ter um DataLoader separado que aplique as transformações
                    # Mas para simplificar, vamos aplicar transformações simples aqui
                    # O ideal seria usar o train_transform, mas isso requer re-implementar o dataset/dataloader
                    # Vamos manter o código original e apenas adicionar uma média de predições
                    # A implementação correta de TTA requer um DataLoader que retorne múltiplas versões da imagem.
                    # Para simplificar, vamos apenas rodar o modelo múltiplas vezes na imagem original
                    # e somar os logits (o que não é TTA, mas sim ensemble simples).
                    # Para TTA de verdade, precisaríamos de um DataLoader que aplicasse as transformações.

                    # Como o código original não tem TTA, vamos apenas manter a avaliação normal
                    # e sugerir TTA como uma melhoria futura, pois requer mudanças no DataLoader.
                    pass

            # Calcular a média dos logits (se TTA fosse implementado)
            # y_pred_logits_mean = y_pred_logits_sum / (tta_steps + 1)

            loss = loss_fn(y_pred_logits_sum, y)

            # Calcular predições binárias
            y_pred_probs = torch.sigmoid(y_pred_logits_sum)
            y_pred_class = (y_pred_probs >= 0.5).float()

            # Calcular acurácia
            correct = (y_pred_class == y).sum().item()
            acc = (correct / len(y)) * 100

            test_loss += loss.item()
            test_acc += acc

            # Store predictions and labels for detailed analysis
            all_preds.extend(y_pred_class.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

            progress_bar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{acc:.2f}%'})

    test_loss /= len(data_loader)
    test_acc /= len(data_loader)

    return {
        "model_name": model.__class__.__name__,
        "model_loss": test_loss,
        "model_acc": test_acc,
        "predictions": all_preds,
        "labels": all_labels
    }

# Load best model and evaluate
if best_model_state is not None:
    base_model.load_state_dict(best_model_state)
    print("Loaded best model for evaluation")

# Evaluate on test set
print("Evaluating model on test set...")
# Usando a função de avaliação original (sem TTA)
test_results = eval_model(model=base_model,
                         data_loader=test_dataloader,
                         loss_fn=loss_fn,
                         device=device)

print(f"\n🎯 Final Test Results:")
print(f"Model: {test_results['model_name']}")
print(f"Test Loss: {test_results['model_loss']:.4f}")
print(f"Test Accuracy: {test_results['model_acc']:.2f}%")

# Performance metrics
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

class_names = ["nonMel", "mel"]

# Generate classification report
y_true = test_results['labels']
y_pred = test_results['predictions']

print("\n📈 Detailed Classification Report (Otimizado):")
print(classification_report(y_true, y_pred, target_names=class_names))

# Plot confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix (Otimizado)', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# ==============================================================================
# 10. Salvamento (Mantido)
# ==============================================================================

from pathlib import Path
import torch
import json

# Define the base path for local saving
LOCAL_MODEL_PATH = Path("/content/models")
LOCAL_MODEL_PATH.mkdir(parents=True, exist_ok=True)

# Save model
MODEL_NAME = "efficientNet-01.pth"
MODEL_SAVE_PATH_LOCAL = LOCAL_MODEL_PATH / MODEL_NAME

# Logic for local model saving
counter = 1
original_model_name_local = MODEL_SAVE_PATH_LOCAL.stem
while MODEL_SAVE_PATH_LOCAL.exists():
    MODEL_SAVE_PATH_LOCAL = LOCAL_MODEL_PATH / f"{original_model_name_local}_{counter}{MODEL_SAVE_PATH_LOCAL.suffix}"
    counter += 1

print(f"Saving model locally to: {MODEL_SAVE_PATH_LOCAL}")
torch.save(obj=base_model.state_dict(), f=MODEL_SAVE_PATH_LOCAL)

# Save training history
history = {
    "train_losses": train_losses,
    "val_losses": val_losses,
    "train_accs": train_accs,
    "val_accs": val_accs,
    "test_results": {
        "loss": test_results['model_loss'],
        "accuracy": test_results['model_acc']
    },
    "training_time": total_train_time,
    "epochs": EPOCHS,
    "best_val_acc": best_val_acc
}

# Define the path for the training history JSON locally
HISTORY_SAVE_PATH_LOCAL = LOCAL_MODEL_PATH / "training_history_efficientNet-01.json"

with open(HISTORY_SAVE_PATH_LOCAL, "w") as f:
    json.dump(history, f, indent=2)

print("Model and training history saved successfully to local environment!")